In [1]:
# ── Universe ──────────────────────────────────────────────────────────────────
SENSEX_30 = [
    'RELIANCE.BO',  'TCS.BO',       'HDFCBANK.BO',  'INFY.BO',      'ICICIBANK.BO',
    'HINDUNILVR.BO','ITC.BO',       'SBIN.BO',      'BHARTIARTL.BO','KOTAKBANK.BO',
    'LT.BO',        'AXISBANK.BO',  'ASIANPAINT.BO','MARUTI.BO',    'SUNPHARMA.BO',
    'TITAN.BO',     'BAJFINANCE.BO','WIPRO.BO',     'ONGC.BO',      'NTPC.BO',
    'POWERGRID.BO', 'ULTRACEMCO.BO','NESTLEIND.BO', 'BAJAJFINSV.BO','TATAMOTORS.NS',
    'HCLTECH.BO',   'TATASTEEL.BO', 'JSWSTEEL.BO',  'M&M.BO',       'ADANIENT.BO',
]
SENSEX_BENCHMARK = '^BSESN'

NIFTY_50 = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
    'LT.NS','AXISBANK.NS','ASIANPAINT.NS','MARUTI.NS','SUNPHARMA.NS',
    'TITAN.NS','BAJFINANCE.NS','WIPRO.NS','ONGC.NS','NTPC.NS',
    'POWERGRID.NS','ULTRACEMCO.NS','NESTLEIND.NS','BAJAJFINSV.NS','TATAMOTORS.NS',
    'HCLTECH.NS','TATASTEEL.NS','JSWSTEEL.NS','M&M.NS','ADANIENT.NS',

    'ADANIPORTS.NS','APOLLOHOSP.NS','BAJAJ-AUTO.NS','BPCL.NS','BRITANNIA.NS',
    'CIPLA.NS','COALINDIA.NS','DIVISLAB.NS','DRREDDY.NS','EICHERMOT.NS',
    'GRASIM.NS','HDFCLIFE.NS','HEROMOTOCO.NS','HINDALCO.NS','INDUSINDBK.NS',
    'JSWSTEEL.NS','LTIM.NS','SBILIFE.NS','SHRIRAMFIN.NS','TECHM.NS',
    'UPL.NS'
]
NIFTY_BENCHMARK = '^NSEI'

# ── Date ranges ───────────────────────────────────────────────────────────────
START_DATE  = '2014-01-01'
END_DATE    = '2024-12-31'
TRAIN_END   = '2021-12-31'
VAL_END     = '2022-12-31'
# Test = 2023-01-01 → 2024-12-31

# ── Feature / model geometry ──────────────────────────────────────────────────
WINDOW          = 10    # W: trading-day lookback window per sample
N_FEATURES      = 7     # F: per-stock features (see engineering cell)
STOCK_EMBED_DIM = 64    # per-stock latent dimension (Stage-1 encoder output)
LATENT_DIM      = 64    # portfolio-level latent dim |z_t|
DROPOUT         = 0.10

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE = 32
N_EPOCHS   = 80
LR         = 3e-4
PATIENCE   = 15

# Loss weights — blueprint: L_recon + α*L_linear + β*L_risk + γ*L_stability
#   L_recon     : autoencoder reconstruction fidelity
#   L_linear    : Koopman linearisation  ← most important, keep highest
#   L_risk      : covariance (Σ) prediction quality
#   L_stability : spectral radius penalty on K
LAMBDA_RECON     = 1.0
LAMBDA_LINEAR    = 4.0   # α
LAMBDA_RISK      = 2.0   # β
LAMBDA_STABILITY = 0.5   # γ

# ── Σ regularisation (PD guarantee in C readout) ──────────────────────────────
# A small ridge added to the symmetrised C output matrix during both training
# (decode_sigma) and inference (_forecast_sigmas) to guarantee positive-definiteness.
SIGMA_REG = 1e-4

# ── MPC (Phase 2) ─────────────────────────────────────────────────────────────
H          = 7       # planning horizon (days)
LAMBDA_TC  = 0.001   # L1 transaction friction coefficient (≈ 10 bps per unit turnover)
MAX_WEIGHT = 0.20    # max single-stock weight (long-only, diversification cap)

# ── Log Return feature index ──────────────────────────────────────────────────
LOGRET_IDX = 0   # feature 0 is log_return (see engineering cell)

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

# ── Data and Benchmarks ───────────────────────────────────────────────────────
SENSEX_30 = [
    'RELIANCE.BO',  'TCS.BO',       'HDFCBANK.BO',  'INFY.BO',      'ICICIBANK.BO',
    'HINDUNILVR.BO','ITC.BO',       'SBIN.BO',      'BHARTIARTL.BO','KOTAKBANK.BO',
    'LT.BO',        'AXISBANK.BO',  'ASIANPAINT.BO','MARUTI.BO',    'SUNPHARMA.BO',
    'TITAN.BO',     'BAJFINANCE.BO','WIPRO.BO',     'ONGC.BO',      'NTPC.BO',
    'POWERGRID.BO', 'ULTRACEMCO.BO','NESTLEIND.BO', 'BAJAJFINSV.BO','TATAMOTORS.NS',
    'HCLTECH.BO',   'TATASTEEL.BO', 'JSWSTEEL.BO',  'M&M.BO',       'ADANIENT.BO',
]
SENSEX_BENCHMARK = '^BSESN'

NIFTY_50 = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
    'LT.NS','AXISBANK.NS','ASIANPAINT.NS','MARUTI.NS','SUNPHARMA.NS',
    'TITAN.NS','BAJFINANCE.NS','WIPRO.NS','ONGC.NS','NTPC.NS',
    'POWERGRID.NS','ULTRACEMCO.NS','NESTLEIND.NS','BAJAJFINSV.NS','TATAMOTORS.NS',
    'HCLTECH.NS','TATASTEEL.NS','JSWSTEEL.NS','M&M.NS','ADANIENT.NS',
    'ADANIPORTS.NS','APOLLOHOSP.NS','BAJAJ-AUTO.NS','BPCL.NS','BRITANNIA.NS',
    'CIPLA.NS','COALINDIA.NS','DIVISLAB.NS','DRREDDY.NS','EICHERMOT.NS',
    'GRASIM.NS','HDFCLIFE.NS','HEROMOTOCO.NS','HINDALCO.NS','INDUSINDBK.NS',
    'JSWSTEEL.NS','LTIM.NS','SBILIFE.NS','SHRIRAMFIN.NS','TECHM.NS',
    'UPL.NS'
]
NIFTY_BENCHMARK = '^NSEI'

# ── Date ranges ───────────────────────────────────────────────────────────────
START_DATE  = '2014-01-01'
END_DATE    = '2024-12-31'
TRAIN_END   = '2021-12-31'
VAL_END     = '2022-12-31'

# ── Helper Functions ──────────────────────────────────────────────────────────

def fetch_data(tickers, start, end):
    """Fetches adjusted close prices and returns a clean DataFrame."""
    # Remove duplicates
    tickers = list(set(tickers))
    print(f"Downloading data for {len(tickers)} tickers...")
    data = yf.download(tickers, start=start, end=end, progress=False)['Adj Close']

    if isinstance(data, pd.Series):
        data = data.to_frame()

    # Drop assets that don't have a full history for the given period
    data = data.dropna(axis=1)
    return data

def get_markowitz_weights(returns):
    """Optimizes portfolio to maximize Sharpe Ratio (Risk-Free Rate = 0 for simplicity)."""
    mean_returns = returns.mean()
    cov_matrix = returns.cov()
    num_assets = len(returns.columns)

    def neg_sharpe(weights):
        p_ret = np.sum(mean_returns * weights) * 252
        p_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))
        return -p_ret / p_vol

    # Constraints: weights sum to 1. Bounds: 0 <= weight <= 1 (long only)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    init_guess = num_assets * [1. / num_assets,]

    result = minimize(neg_sharpe, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
    return result.x

def calculate_metrics(returns_series, name):
    """Calculates CAGR, Sharpe, Calmar, Win%, and Max Drawdown."""
    cum_returns = (1 + returns_series).cumprod()

    # CAGR
    total_return = cum_returns.iloc[-1] - 1
    years = len(returns_series) / 252.0
    cagr = (1 + total_return) ** (1 / years) - 1

    # Sharpe
    sharpe = np.sqrt(252) * returns_series.mean() / returns_series.std()

    # Max Drawdown
    rolling_max = cum_returns.cummax()
    drawdown = cum_returns / rolling_max - 1
    max_dd = abs(drawdown.min())

    # Calmar
    calmar = cagr / max_dd if max_dd != 0 else np.nan

    # Win% (Daily)
    win_pct = (returns_series > 0).sum() / len(returns_series)

    print(f"--- {name} ---")
    print(f"CAGR:         {cagr * 100:.2f}%")
    print(f"Sharpe Ratio: {sharpe:.2f}")
    print(f"Max Drawdown: {max_dd * 100:.2f}%")
    print(f"Calmar Ratio: {calmar:.2f}")
    print(f"Win%:         {win_pct * 100:.2f}%\n")

# ── Main Execution ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Choose portfolio to test (using SENSEX_30 in this example)
    tickers = SENSEX_30
    benchmark_ticker = SENSEX_BENCHMARK

    # 1. Fetch Data
    print("Fetching Asset Data...")
    prices = fetch_data(tickers, START_DATE, END_DATE)
    daily_returns = prices.pct_change().dropna()

    print("Fetching Benchmark Data...")
    bench_prices = fetch_data([benchmark_ticker], START_DATE, END_DATE)
    bench_returns = bench_prices.pct_change().dropna().squeeze()

    # 2. Split Data (Train vs Out-of-Sample Test)
    # Using Train period to find weights, and Test period to evaluate performance
    train_returns = daily_returns[:TRAIN_END]
    test_returns = daily_returns[TRAIN_END:END_DATE]
    bench_test_returns = bench_returns[TRAIN_END:END_DATE]

    # 3. Optimize Portfolio
    print("Running Markowitz Mean-Variance Optimization on Train Data...")
    optimal_weights = get_markowitz_weights(train_returns)

    # Show Top 5 allocations
    weights_series = pd.Series(optimal_weights, index=train_returns.columns).sort_values(ascending=False)
    print("\nTop 5 Portfolio Allocations:")
    for asset, weight in weights_series.head(5).items():
        if weight > 0.01:
            print(f"  {asset}: {weight * 100:.2f}%")
    print("\n")

    # 4. Apply Weights to Out-of-Sample (Test) Data
    portfolio_test_returns = test_returns.dot(optimal_weights)

    # Align dates between portfolio and benchmark
    aligned_data = pd.DataFrame({
        'Portfolio': portfolio_test_returns,
        'Benchmark': bench_test_returns
    }).dropna()

    # 5. Evaluate and Print Metrics
    print(f"OUT-OF-SAMPLE PERFORMANCE ({TRAIN_END} to {END_DATE})\n" + "="*50)
    calculate_metrics(aligned_data['Portfolio'], "Markowitz Optimized Portfolio")
    calculate_metrics(aligned_data['Benchmark'], "Benchmark")

Fetching Asset Data...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


Fetching Benchmark Data...


KeyError: 'Adj Close'